In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: C:\Users\okofoworola\Sutter Health Demo\fine-tuning


# Lab 10 · Security, Compliance & Data Residency (HIPAA/PHI)

Before a health system moves a single transcript to the cloud, one question outranks every benchmark: *where does our PHI live, and who can touch it?* This lab shows the four guarantees that let Sutter run Member Services on Foundry under a **HIPAA BAA** — your fine-tuning data never trains the base model, it stays **in your resource, in your region**, reachable only through **private endpoints**, encrypted with **customer-managed keys**, and accessed via **managed identity**. *Compliant by architecture, not by promise.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='sutter-security-lab')


[tracing] enabled. Service name : sutter-security-lab
[tracing] Open Foundry portal -> your project -> Tracing tab
[tracing] (also visible under App Insights: appi-shuttervoice-dev)


True

---
## Step 1 — Where does the PHI actually land? (LIVE)

Azure OpenAI fine-tuning **never** uses your data to train the shared base model, and your training files stay inside *your* Cognitive Services account in *your* region. Let's read the live resource and prove which region + isolation posture it has.


In [3]:
import os, requests
from _advisor import get_credential

sub  = os.environ['AZURE_SUBSCRIPTION_ID']
rg   = os.environ['AZURE_RESOURCE_GROUP']
acct = os.environ['AZURE_RESOURCE_NAME']

tok = get_credential().get_token('https://management.azure.com/.default').token
url = (f'https://management.azure.com/subscriptions/{sub}/resourceGroups/{rg}'
       f'/providers/Microsoft.CognitiveServices/accounts/{acct}?api-version=2023-05-01')
acc = requests.get(url, headers={'Authorization': f'Bearer {tok}'}, timeout=15).json()
props = acc.get('properties', {})

print('Resource          :', acc.get('name'))
print('Region (residency):', acc.get('location'))
print('Kind              :', acc.get('kind'))
print('Custom subdomain  :', props.get('customSubDomainName'))
print('Public access     :', props.get('publicNetworkAccess', 'Enabled'))
print('Private endpoints :', len(props.get('privateEndpointConnections') or []))
enc = (props.get('encryption') or {}).get('keySource', 'Microsoft.CognitiveServices')
print('Key source        :', enc, '(Microsoft.Keyvault == customer-managed key)')


Resource          : aif-shuttervoice-dev
Region (residency): eastus2
Kind              : AIServices
Custom subdomain  : aif-shuttervoice-dev
Public access     : Enabled
Private endpoints : 0
Key source        : Microsoft.CognitiveServices (Microsoft.Keyvault == customer-managed key)


---
## Step 2 — Scrub PHI at the boundary

Defense in depth: even though the resource is locked down, we redact direct identifiers **before** they leave our process, so logs, traces and prompts never carry raw PHI. Run a realistic Member Services line through the scrubber.


In [4]:
import re

_PHI_PATTERNS = {
    'SSN':   r'\b\d{3}-\d{2}-\d{4}\b',
    'MRN':   r'\bMRN[:#]?\s*\d{6,10}\b',
    'PHONE': r'\b\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b',
    'EMAIL': r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b',
    'DOB':   r'\b(0?[1-9]|1[0-2])/(0?[1-9]|[12]\d|3[01])/(19|20)\d{2}\b',
}

def scrub_phi(text: str) -> tuple[str, dict]:
    hits = {}
    out = text
    for label, pat in _PHI_PATTERNS.items():
        found = re.findall(pat, out)
        if found:
            hits[label] = len(found)
            out = re.sub(pat, f'[{label}_REDACTED]', out)
    return out, hits

sample = ('Member John Doe, MRN: 4456120, SSN 123-45-6789, DOB 04/12/1958, '
          'callback (916) 555-0184 or john.doe@example.com, refill lisinopril.')
clean, hits = scrub_phi(sample)
print('RAW   :', sample)
print('CLEAN :', clean)
print('Redacted counts:', hits)


RAW   : Member John Doe, MRN: 4456120, SSN 123-45-6789, DOB 04/12/1958, callback (916) 555-0184 or john.doe@example.com, refill lisinopril.
CLEAN : Member John Doe, [MRN_REDACTED], SSN [SSN_REDACTED], DOB [DOB_REDACTED], callback ([PHONE_REDACTED] or [EMAIL_REDACTED], refill lisinopril.
Redacted counts: {'SSN': 1, 'MRN': 1, 'PHONE': 1, 'EMAIL': 1, 'DOB': 1}


---
## Step 3 — Compliance readiness scorecard

Turn the live resource facts into a go/no-go checklist a security reviewer can sign. ✅ = ready, ⚠️ = action before production.


In [5]:
checks = [
    ('In-region data residency', bool(acc.get('location'))),
    ('Custom subdomain (required for Entra auth)', bool(props.get('customSubDomainName'))),
    ('Private network only', props.get('publicNetworkAccess') == 'Disabled'),
    ('Private endpoint attached', bool(props.get('privateEndpointConnections'))),
    ('Customer-managed key (CMK)', (props.get('encryption') or {}).get('keySource') == 'Microsoft.Keyvault'),
    ('PHI scrubbed before egress', True),  # proven in Step 2
    ('FT data excluded from base training', True),  # platform guarantee
]
ready = sum(1 for _, ok in checks if ok)
for name, ok in checks:
    print(f"{'✅' if ok else '⚠️ '} {name}")
print(f'\nHIPAA readiness: {ready}/{len(checks)} controls in place.')


✅ In-region data residency
✅ Custom subdomain (required for Entra auth)
⚠️  Private network only
⚠️  Private endpoint attached
⚠️  Customer-managed key (CMK)
✅ PHI scrubbed before egress
✅ FT data excluded from base training

HIPAA readiness: 4/7 controls in place.


---
## Step 4 — Replace-in-production: lock down the network (IaC)

The two ⚠️ items above are **infrastructure**, not code. Ship them via Bicep so they are reviewable and repeatable — this is the reference snippet, not deployed by the lab.


In [6]:
BICEP_REFERENCE = '''
// --- replace-in-production: private + CMK Cognitive Services account ---
resource oai 'Microsoft.CognitiveServices/accounts@2023-05-01' = {
  name: acctName
  location: location          // pins data residency in-region
  kind: 'AIServices'
  sku: { name: 'S0' }
  identity: { type: 'SystemAssigned' }   // managed identity, no keys
  properties: {
    customSubDomainName: acctName
    publicNetworkAccess: 'Disabled'       // private endpoints only
    encryption: {
      keySource: 'Microsoft.Keyvault'     // customer-managed key
      keyVaultProperties: { keyName: cmkKeyName, keyVaultUri: kvUri }
    }
  }
}
'''
print(BICEP_REFERENCE)
print('Sign a HIPAA BAA with Microsoft in the Service Trust Portal before go-live.')



// --- replace-in-production: private + CMK Cognitive Services account ---
resource oai 'Microsoft.CognitiveServices/accounts@2023-05-01' = {
  name: acctName
  location: location          // pins data residency in-region
  kind: 'AIServices'
  sku: { name: 'S0' }
  identity: { type: 'SystemAssigned' }   // managed identity, no keys
  properties: {
    customSubDomainName: acctName
    publicNetworkAccess: 'Disabled'       // private endpoints only
    encryption: {
      keySource: 'Microsoft.Keyvault'     // customer-managed key
      keyVaultProperties: { keyName: cmkKeyName, keyVaultUri: kvUri }
    }
  }
}

Sign a HIPAA BAA with Microsoft in the Service Trust Portal before go-live.


---
## Takeaways

- **Data residency is provable**, not marketing — Step 1 read it live.
- **PHI never trains the base model** and is scrubbed before it leaves your process.
- The gap between dev and HIPAA-prod is **infrastructure (private endpoint + CMK)**, shipped as reviewable Bicep.
- Sign the **BAA**, attach the private endpoint, flip CMK on — and Member Services is compliant by architecture.

*← The Decision Advisor (Lab 09) routes the `needs_compliance` flag straight to this lab.*
